# 03 — Stage 1: YOLOv8 Detector (DENTEX 4 kelas penyakit)

Detektor YOLO untuk bbox + diagnosis (Impacted/Caries/Periapical/Deep Caries). **Plan B resmi** roadmap (weights pretrained HierarchicalDet tidak dirilis publik). Selaras dengan pendekatan supervisor pada panoramic radiograph — **Veerabhadrappa & Vengusamy (2025) memakai YOLOv7**; di sini diperbarui ke **YOLOv8**.

Output: `checkpoints/yolov8_dentex.pt` di Drive. Runtime: **GPU**.

## Cell 1 — Mount Drive + sync repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/opg-live'

import os
REPO = '/content/opg-live'
if not os.path.exists(REPO):
    !git clone https://github.com/AndreasTopuh/opg-live.git {REPO}
!cd {REPO} && git fetch origin && git reset --hard origin/main && git log --oneline -1

## Cell 2 — Install Ultralytics (YOLOv8)

In [ ]:
%pip install -q ultralytics
import torch; from ultralytics import YOLO
print('GPU:', torch.cuda.get_device_name(0))

## Cell 3 — Konversi DENTEX → format YOLO
Copy gambar + tulis label ke `/content/yolo` (lokal Colab = cepat). Split per-gambar.

In [ ]:
!python /content/opg-live/scripts/dentex_to_yolo.py --drive {DRIVE_ROOT} --out /content/yolo
!echo '--- contoh label ---'; head -3 $(ls /content/yolo/labels/train/*.txt | head -1)

## Cell 4 — Train YOLOv8m
`imgsz=1024` karena lesi karies kecil di OPG lebar (640 terlalu kecil). Turunkan `batch` kalau OOM.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8m.pt')
results = model.train(
    data='/content/yolo/dentex.yaml',
    epochs=80, imgsz=1024, batch=16,
    project='/content/runs', name='dentex', exist_ok=True,
    patience=20,  # early stopping kalau val mAP stuck 20 epoch
)

## Cell 5 — Simpan best.pt ke Drive (aman dari reset Colab)

In [ ]:
import shutil
src = '/content/runs/dentex/weights/best.pt'
dst = f'{DRIVE_ROOT}/checkpoints/yolov8_dentex.pt'
shutil.copy(src, dst)
print('✅ saved', dst)

## Cell 6 — Evaluasi: mAP global + per-kelas
Target: ≥ baseline HierarchicalDet (Stage 1 = reuse, bukan novelty). Disease AR HD ~0.69.

In [ ]:
from ultralytics import YOLO
model = YOLO(f'{DRIVE_ROOT}/checkpoints/yolov8_dentex.pt')
m = model.val(data='/content/yolo/dentex.yaml', imgsz=1024)
print('mAP50    :', round(float(m.box.map50), 4))
print('mAP50-95 :', round(float(m.box.map), 4))
names = m.names
for i, ap in enumerate(m.box.maps):
    print(f'  {names[i]:20s} mAP50-95 {ap:.4f}')

## Cell 7 — Preview deteksi 1 gambar val (sanity check)

In [ ]:
import glob, matplotlib.pyplot as plt, cv2
img = sorted(glob.glob('/content/yolo/images/val/*.png'))[0]
res = model.predict(img, imgsz=1024, conf=0.25)[0]
plt.figure(figsize=(16, 7))
plt.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
for b in res.boxes:
    print(res.names[int(b.cls)], 'conf', round(float(b.conf), 3), 'xyxy', [round(v) for v in b.xyxy[0].tolist()])

## ✅ Deliverable Stage 1
- [ ] `yolov8_dentex.pt` di Drive
- [ ] mAP50 / mAP50-95 global + per-kelas terlaporkan
- [ ] preview deteksi terlihat masuk akal

**Catatan tulisan:** Stage 1 = detektor YOLO, konsisten dengan pendekatan supervisor pada panoramic radiograph (Veerabhadrappa & Vengusamy, 2025; YOLOv7), diperbarui ke YOLOv8. Reuse, bukan novelty.

**Next:** rewire `make_artifacts.py` agar bbox prompt diambil dari **deteksi YOLO Stage 1** (bukan GT) → 3-arm artifact → Stage 3 (GPT-4o + RAG).